Data Preparation

In [1]:
import pandas as pd 
import re

In [2]:
# load data 
df = pd.read_csv("cleaned_data/bangalore_cleaned.csv")
df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
3,Super built-up Area,Ready To Move,Whitefield,2 BHK,DuenaTa,1170,2.0,1.0,38.00
4,Plot Area,Ready To Move,Whitefield,4 Bedroom,Prrry M,2785,5.0,3.0,295.00


In [3]:
df.isnull().sum()

area_type       0
availability    0
location        0
size            0
society         0
total_sqft      0
bath            0
balcony         0
price           0
dtype: int64

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7144 entries, 0 to 7143
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     7144 non-null   object 
 1   availability  7144 non-null   object 
 2   location      7144 non-null   object 
 3   size          7144 non-null   object 
 4   society       7144 non-null   object 
 5   total_sqft    7144 non-null   object 
 6   bath          7144 non-null   float64
 7   balcony       7144 non-null   float64
 8   price         7144 non-null   float64
dtypes: float64(3), object(6)
memory usage: 502.4+ KB


size is object type and total_sqft is also object type.

We will convert total_sqft to numeric and for different formats present in this feature eg 2100 - 2200 , we will take the average of the two numbers and convert the entire column to numeric type.

We will Extract BHK from the size column and this feature also has two unique formats eg. 2 BHK , 2 Bedroom. We will consider both of them as 2 BHK and convert the entire column to numeric type.

In [5]:
# function to convert the total_sqft column to numeric

def convert_sqft(x):
    try:
        s = str(x).strip()
        if "-" in s:
            parts = s.split("-")
            return (float(parts[0]) + float(parts[1]))/2
        return float(s)

    except: # if parsing fails, return None
        return None

df["total_sqft"] = df["total_sqft"].apply(convert_sqft)

In [6]:
df.isnull().sum()

area_type        0
availability     0
location         0
size             0
society          0
total_sqft      15
bath             0
balcony          0
price            0
dtype: int64

In [7]:
# Dropping the rows with null values in total_sqft column
df = df.dropna(subset=["total_sqft"])

In [8]:
# Function to extract BHK from the size column

def extract_bhk(x):
    if pd.isna(x): return None
    m = re.search(r"(\d+)\s*(bhk|bedroom)" , str(x).lower())

    if m:
        return int(m.group(1)) # return the number part as integer

    return None


df['bhk'] = df['size'].apply(extract_bhk)

In [9]:
df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price,bhk
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056.0,2.0,1.0,39.07,2.0
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600.0,5.0,3.0,120.00,4.0
2,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521.0,3.0,1.0,95.00,3.0
3,Super built-up Area,Ready To Move,Whitefield,2 BHK,DuenaTa,1170.0,2.0,1.0,38.00,2.0
4,Plot Area,Ready To Move,Whitefield,4 Bedroom,Prrry M,2785.0,5.0,3.0,295.00,4.0


In [10]:
df.isnull().sum()

area_type        0
availability     0
location         0
size             0
society          0
total_sqft       0
bath             0
balcony          0
price            0
bhk             10
dtype: int64

In [11]:
# drop the rows with null values in bhk column
df = df.dropna(subset=["bhk"])

In [12]:
# drop size column as we have extracted bhk from it

df = df.drop("size" , axis=1)

In [13]:
df['availability'].value_counts()

availability
Ready To Move    5441
18-May            167
18-Dec            164
19-Dec            147
18-Apr            146
                 ... 
16-Nov              1
22-Jan              1
16-Jan              1
17-Feb              1
14-Jul              1
Name: count, Length: 74, dtype: int64

In [14]:
# Replacing the string Ready To Move with Immediate Possession in availability column
df['availability'] = df['availability'].replace("Ready To Move", "Immediate Possession")

In [15]:
# Create a new column to provide descriptive text for each property listing

df['text'] = (
    "A spacious " + df['bhk'].astype(str) + " BHK property located in " + df['location'] + 
    ", offering " + df['total_sqft'].astype(str) + " sqft of " + df['area_type'].astype(str) + 
    " in " + df['society'] + " society, priced at " + df['price'].astype(str) + " lakh. " +
    "This property features " + df['bath'].astype(int).astype(str) + " bathroom(s) and " + 
    df['balcony'].astype(int).astype(str) + " balcony(ies), with availability status: " + 
    df['availability'] + "."
)

In [16]:
df.head()

,area_type,availability,location,society,total_sqft,bath,balcony,price,bhk,text
0,Super built-up Area,19-Dec,Electronic City Phase II,Coomee,1056.0,2.0,1.0,39.07,2.0,A spacious 2.0 BHK property located in Electro...
1,Plot Area,Immediate Possession,Chikka Tirupathi,Theanmp,2600.0,5.0,3.0,120.00,4.0,A spacious 4.0 BHK property located in Chikka ...
2,Super built-up Area,Immediate Possession,Lingadheeranahalli,Soiewre,1521.0,3.0,1.0,95.00,3.0,A spacious 3.0 BHK property located in Lingadh...
3,Super built-up Area,Immediate Possession,Whitefield,DuenaTa,1170.0,2.0,1.0,38.00,2.0,A spacious 2.0 BHK property located in Whitefi...
4,Plot Area,Immediate Possession,Whitefield,Prrry M,2785.0,5.0,3.0,295.00,4.0,A spacious 4.0 BHK property located in Whitefi...


In [17]:
# see full text
pd.set_option('display.max_colwidth', None) 
 
df['text']

0              A spacious 2.0 BHK property located in Electronic City Phase II, offering 1056.0 sqft of Super built-up  Area in Coomee  society, priced at 39.07 lakh. This property features 2 bathroom(s) and 1 balcony(ies), with availability status: 19-Dec.
1                  A spacious 4.0 BHK property located in Chikka Tirupathi, offering 2600.0 sqft of Plot  Area in Theanmp society, priced at 120.0 lakh. This property features 5 bathroom(s) and 3 balcony(ies), with availability status: Immediate Possession.
2       A spacious 3.0 BHK property located in Lingadheeranahalli, offering 1521.0 sqft of Super built-up  Area in Soiewre society, priced at 95.0 lakh. This property features 3 bathroom(s) and 1 balcony(ies), with availability status: Immediate Possession.
3               A spacious 2.0 BHK property located in Whitefield, offering 1170.0 sqft of Super built-up  Area in DuenaTa society, priced at 38.0 lakh. This property features 2 bathroom(s) and 1 balcony(ies), with availabilit

In [18]:
# return back to original max colwidth
# pd.set_option('display.max_colwidth', 50)

In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7119 entries, 0 to 7143
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     7119 non-null   object 
 1   availability  7119 non-null   object 
 2   location      7119 non-null   object 
 3   society       7119 non-null   object 
 4   total_sqft    7119 non-null   float64
 5   bath          7119 non-null   float64
 6   balcony       7119 non-null   float64
 7   price         7119 non-null   float64
 8   bhk           7119 non-null   float64
 9   text          7119 non-null   object 
dtypes: float64(5), object(5)
memory usage: 611.8+ KB


In [20]:
# find the maxumum length of the text column
max_length = df['text'].apply(len).max()
print("Maximum length of text column:", max_length)


# find the max number of words in the text column
max_words = df['text'].apply(lambda x: len(str(x).split()))
print("Maximum number of words in text column:", max_words.max())

Maximum length of text column: 262
Maximum number of words in text column: 39


In [ ]:
# Creating embedding for each row in the text column using Hugging Face's sentence-transformers library

import numpy as np
from sentence_transformers import SentenceTransformer
import faiss 


# # Prepare texts and IDs 

# texts = df['text'].tolist()
# ids = np.arange(len(texts)).astype('int64')

# # Create embeddings 

# embed_model = SentenceTransformer('all-MiniLM-L6-v2') # small and fast for generating embeddings

# embeddings = embed_model.encode(texts,show_progress_bar=True, convert_to_numpy=True)

# # Normalize the embeddings (This helps with the cosine similarity search)

# faiss.normalize_L2(embeddings)

# # Build Faiss index 

# d = embeddings.shape[1] # dimension of the embeddings
# cpu_index = faiss.IndexFlatIP(d) # inner product on normalized vectors is equivalent to cosine similarity

# # move to gpu 
# res = faiss.StandardGpuResources() # Initialize GPU resources
# index = faiss.index_cpu_to_gpu(res, 0, cpu_index) # move the index to GPU


# index.add(embeddings) # add the embeddings to the index

# print(f"Index is on GPU: {index.is_trained} and has {index.ntotal} vectors.")

d:\apps\InstalledApplications\anaconda3\envs\realestate_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 736.17it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 223/223 [00:59<00:00,  3.76it/s]


Index is on GPU: True and has 7119 vectors.


In [22]:
# # Save artifacts to disk to avoid regenerating embeddings each run
# import pickle

# # 1. Save embeddings (numpy array) - useful for rebuilding or analysis
# with open("embeddings.pkl", "wb") as f:
#     pickle.dump(embeddings, f)

# # 2. Save the FAISS index - this is what you actually search
# # Note: GPU index needs to be moved back to CPU before saving
# cpu_index_to_save = faiss.index_gpu_to_cpu(index)
# faiss.write_index(cpu_index_to_save, "faiss_index.bin")

# # 3. Save the texts list - needed to map index IDs back to property descriptions
# with open("texts.pkl", "wb") as f:
#     pickle.dump(texts, f)

# print("Saved: embeddings.pkl, faiss_index.bin, texts.pkl")

Saved: embeddings.pkl, faiss_index.bin, texts.pkl


**Loading Saved Artifacts (Alternative to regenerating embeddings)**

If you've already saved the index .This is to skip the embedding generation step, uncomment and run the cell below instead of the embedding generation cell.

In [21]:
# Uncomment to LOAD saved artifacts instead of regenerating
import pickle
import faiss
from sentence_transformers import SentenceTransformer

# Load the texts
with open("texts.pkl", "rb") as f:
    texts = pickle.load(f)

# Load embeddings (optional - only needed if you want to inspect them)
with open("embeddings.pkl", "rb") as f:
    embeddings = pickle.load(f)

# Load the FAISS index
cpu_index = faiss.read_index("faiss_index.bin")

# Move to GPU
res = faiss.StandardGpuResources()
index = faiss.index_cpu_to_gpu(res, 0, cpu_index)

# Load the embedding model (still needed for encoding queries)
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

print(f"Loaded index with {index.ntotal} vectors from disk.")

d:\apps\InstalledApplications\anaconda3\envs\realestate_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 801.76it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded index with 7119 vectors from disk.


In [22]:
# Retrieval Function

def retrieve(query , top_k=5):
    q_emb = embed_model.encode([query] , convert_to_numpy=True) 
    faiss.normalize_L2(q_emb) # normalize the query embedding
    D, I = index.search(q_emb , top_k) # search the index for the top_k most similar vectors
    results = []

    for idx , score in zip(I[0],D[0]):
        if idx == -1: # if no more results
            continue 
        results.append({'id':int(idx) ,
                         'score' : float(score) ,
                           'text' : texts[int(idx)]
                           }
                      )
        
    return results

In [40]:
test_query = "Looking for a 2 BHK apartment in Whitefield with around 1000 sqft area "

results = retrieve(test_query, top_k=5)
results

[{'id': 2094,
  'score': 0.7286970615386963,
  'text': 'A spacious 2.0 BHK property located in Whitefield, offering 1132.0 sqft of Super built-up  Area in Preaunt society, priced at 79.28 lakh. This property features 2 bathroom(s) and 1 balcony(ies), with availability status: 21-Dec.'},
 {'id': 4402,
  'score': 0.7279155254364014,
  'text': 'A spacious 2.0 BHK property located in Whitefield, offering 1356.0 sqft of Super built-up  Area in Preaunt society, priced at 94.84 lakh. This property features 2 bathroom(s) and 1 balcony(ies), with availability status: 21-Dec.'},
 {'id': 4709,
  'score': 0.7181288599967957,
  'text': 'A spacious 2.0 BHK property located in Whitefield, offering 1227.0 sqft of Super built-up  Area in Iseenst society, priced at 70.0 lakh. This property features 2 bathroom(s) and 2 balcony(ies), with availability status: Immediate Possession.'},
 {'id': 3015,
  'score': 0.7167584896087646,
  'text': 'A spacious 2.0 BHK property located in Whitefield, offering 1227.0 

In [25]:
import os 
os.environ['HF_HOME'] = "D:/hf_cache"

In [41]:
# Simple response generation using retrieved context : text generation model - Qwen/Qwen2.5-1.5B-Instruct 
from transformers import pipeline 

model_id = 'Qwen/Qwen2.5-1.5B-Instruct'
# tokenizer = AutoTokenizer.from_pretrained(model_id)

gen = pipeline('text-generation' ,
               model =  model_id ,
                 device = 0 ,
                 dtype = 'auto',
                 model_kwargs = {'cache_dir': "D:/hf_cache"})


def answer_query(query , top_k = 5 , max_new_tokens = 500):


    retrieved = retrieve(query , top_k = top_k)
    
    # Build a concise context string from top results
    context = "\n\n".join([f"Listing {r['id']} : {r['text']}" for r in retrieved])

    prompt = f"""You are a professional real estate assistant.

    Use ONLY the listings provided in the context.

    If listings satisfy user query, recommend them.Provide the complete information about the satisfying listings.
    If none satisfy, say "No exact matches found".But if available try to suggest the closest match.
    Important Notice : Exact matches normally needs to be based on location , price and bhk unless specifically asked by the user for matches based on other factors in the query.
    Context:
    {context}

    User Query: {query}

    """

    out = gen(prompt , max_new_tokens = max_new_tokens , do_sample = False , return_full_text = False)
    # return full text = False ensures we only get the generated answer without the prompt
    return {'response':out[0]['generated_text'].strip() , 'retrieved':retrieved}

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 731.23it/s, Materializing param=model.norm.weight]                              


In [42]:
# Implementing on the test query 

result = answer_query(test_query)

Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [43]:
print(test_query)

Looking for a 2 BHK apartment in Whitefield with around 1000 sqft area 


In [44]:
print(result['response'])

Recommendation: 
     Based on your requirements, here is the listing that best fits:

     **Listing 4402**
     
     - Location: Whitefield
     - Number of Bedrooms (BHK): 2
     - Total Built-Up Area: 1356.0 sqft
     - Price: ₹94.84 Lakh
     - Bathrooms: 2
     - Balcony: 1

     This property offers an ideal balance between size and budget, making it suitable for someone looking for a 2 BHK apartment in Whitefield with approximately 1000 sqft area. It's also within reach of immediate possession, which could be beneficial depending on your timeline. Please note that this listing has availability until December 21st. 

     If you have any further preferences or need more details, feel free to ask! 

     Best regards,
     [Your Name]  
     Real Estate Assistant

Note: The recommendation was made based on the total built-up area being close to the target value of 1000 sqft. However, since the actual area offered in all listed properties exceeds the target, no exact match was fo

In [45]:
result['retrieved']

[{'id': 2094,
  'score': 0.7286970615386963,
  'text': 'A spacious 2.0 BHK property located in Whitefield, offering 1132.0 sqft of Super built-up  Area in Preaunt society, priced at 79.28 lakh. This property features 2 bathroom(s) and 1 balcony(ies), with availability status: 21-Dec.'},
 {'id': 4402,
  'score': 0.7279155254364014,
  'text': 'A spacious 2.0 BHK property located in Whitefield, offering 1356.0 sqft of Super built-up  Area in Preaunt society, priced at 94.84 lakh. This property features 2 bathroom(s) and 1 balcony(ies), with availability status: 21-Dec.'},
 {'id': 4709,
  'score': 0.7181288599967957,
  'text': 'A spacious 2.0 BHK property located in Whitefield, offering 1227.0 sqft of Super built-up  Area in Iseenst society, priced at 70.0 lakh. This property features 2 bathroom(s) and 2 balcony(ies), with availability status: Immediate Possession.'},
 {'id': 3015,
  'score': 0.7167584896087646,
  'text': 'A spacious 2.0 BHK property located in Whitefield, offering 1227.0 

In [48]:
# Example queries and display result

example_queries = [
    "Show me three 3-BHK properties in Whitefield under 80 lakhs",
    "Find 2 BHK apartments in Koramangala with at least 1000 sqft area and price under 1.2 crore.Show first those properties where the price per sqft is the lowest.",
    "List affordable 1 BHK options in Jayanagar under 50 lakhs.Show first those properties where the sqft is the highest but do not cross the budget of 50 lakhs."
]



In [49]:
# LLM output for each example_queries
for q in example_queries:
    print(f"Query: {q}")
    res = answer_query(q)
    print(f"Response: {res['response']}")
    print(f"Retrieved Listings: {[r['text'] for r in res['retrieved']]}")
    print("-"*50)

Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Query: Show me three 3-BHK properties in Whitefield under 80 lakhs


Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Response: Based on your criteria, I would recommend listing number 6104 as it meets all the requirements you specified - it is a 3-BHK property in Whitefield, offers 1634.0 sqft of built-up area, has an immediate possession status, and is priced at 68.0 lakh which is well below the 80-lakh limit you mentioned. 

Please let me know if there's anything else I can assist you with! No exact matches found. However, listing number 6104 seems like the best fit according to your specifications. It is a 3-BHK property in Whitefield that fits within your budget range. Would you like more details or assistance with this property? Let me know how I can help further.
Retrieved Listings: ['A spacious 3.0 BHK property located in Whitefield, offering 1634.0 sqft of Built-up  Area in CMionsy society, priced at 68.0 lakh. This property features 3 bathroom(s) and 2 balcony(ies), with availability status: Immediate Possession.', 'A spacious 3.0 BHK property located in Whitefield, offering 2225.0 sqft of S

Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Response: Recommendation:

Based on your criteria, I would recommend listing 352 as it meets all the requirements you specified - it's a 2 BHK apartment in Koramangala with an area of 1253 sqft, priced at 102.0 lakh (which is less than 1.2 crore). It also has 2 bathrooms and 1 balcony, and its availability status is immediate possession. The price per square foot is calculated as 102,000 / 1253 = approximately 81.03 rupees/sq ft, which is indeed the lowest among the listed options. 

Please note that while this listing satisfies most of your conditions, there might be other properties that better fit your specific preferences or budget. However, based solely on the given data, listing 352 stands out as the best option within the constraints you've set. No exact matches were found for your query due to the lack of additional details such as location, price range, etc., but this recommendation should help guide you towards finding suitable properties. Let me know if you need any further 